In [ ]:
sentences = ['bob likes sheep', 'alice is fast', 'cs685 is fun', 'i love lamp']


In [ ]:
# given the first two words of each sentence, we will try to predict the
# third word using a fixed window NLM

# before we start any fancy modeling, we have to tokenize our input
# and convert the words to indices

vocab = {} # map from word type to index
inputs = [] # stores an indexified version of each sentence

for sent in sentences:
    sent_idxes = []
    sent = sent.split() # tokenize w/ whitespace
    for w in sent:
        if w not in vocab:
            vocab[w] = len(vocab) # add a new type to the vocab
        sent_idxes.append(vocab[w])
    inputs.append(sent_idxes)

print(vocab)
print(inputs)

{'bob': 0, 'likes': 1, 'sheep': 2, 'alice': 3, 'is': 4, 'fast': 5, 'cs685': 6, 'fun': 7, 'i': 8, 'love': 9, 'lamp': 10}
[[0, 1, 2], [3, 4, 5], [6, 4, 7], [8, 9, 10]]


In [ ]:
import torch
# two things: 1. convert to LongTensors, 2. define inputs / outputs
prefixes = torch.LongTensor([sent[:-1] for sent in inputs])
labels = torch.LongTensor([sent[-1] for sent in inputs])

torch.Size([4, 2])


In [ ]:
# onto defining the network!
import torch.nn as nn

class NLM(nn.Module):
    # two things that you need to do
    # 1. init function (initializes all the **parameters** of the network)
    # 2. forward function (defines the forward propagation computations)

    def __init__(self, d_embedding, d_hidden, window_size, len_vocab):
        super(NLM, self).__init__() # init the base Module class
        self.d_emb = d_embedding
        self.embeddings = nn.Embedding(len_vocab, d_embedding)

        # concatenated embeddings > hidden
        self.W_hid = nn.Linear(d_embedding*window_size, d_hidden)

        # hidden > output probability distribution over vocab
        self.W_out = nn.Linear(d_hidden, len_vocab)

    def forward(self, input): # each input will be a batch of prefixes (in this case 4)
        batch_size, window_size = input.size()
        embs = self.embeddings(input) # 4 x 2 x 5
        # print('embedding size:', embs.size())

        # next,  we want to concatenate the prefix embeddings together
        concat_embs = embs.view(batch_size, window_size * self.d_emb)
        # print('concatenated embs size:', concat_embs.size())

        # now, we project this to the hidden space
        hiddens = self.W_hid(concat_embs)
        # print('hidden size:', hiddens.size())

        # finally, project hiddens to vocabulary space
        outs = self.W_out(hiddens)
        # print('output size:', outs.size())

        return outs # return unnormalized probability, also known as **logits**


network = NLM(d_embedding=5, d_hidden=12, window_size=2, len_vocab=len(vocab))

num_epochs = 100 # how many times we are going to go through our entire training dataset
learning_rate = 0.1 # we can modify this if training isn't converging, etc.
loss_fn = nn.CrossEntropyLoss() # average cross entropy loss over each prefix in batch

# we will use vanilla gradient descent, you can experiment with others like Adam
optimizer = torch.optim.SGD(params=network.parameters(), lr=learning_rate)

# training loop
for i in range(num_epochs):
    logits = network(prefixes)
    loss = loss_fn(logits, labels)
    # now let's update our params to make the loss smaller
    # step 1: compute gradient (partial derivs of loss WRT parameters)
    loss.backward()

    # step 2: update params using gradient descent
    optimizer.step()

    # step 3: zero out the gradients for the next epoch
    optimizer.zero_grad()
    print('epoch: %d, loss: %0.2f' % (i, loss))




epoch: 0, loss: 2.71
epoch: 1, loss: 2.50
epoch: 2, loss: 2.32
epoch: 3, loss: 2.16
epoch: 4, loss: 2.00
epoch: 5, loss: 1.85
epoch: 6, loss: 1.71
epoch: 7, loss: 1.58
epoch: 8, loss: 1.45
epoch: 9, loss: 1.32
epoch: 10, loss: 1.21
epoch: 11, loss: 1.10
epoch: 12, loss: 1.01
epoch: 13, loss: 0.92
epoch: 14, loss: 0.84
epoch: 15, loss: 0.77
epoch: 16, loss: 0.70
epoch: 17, loss: 0.64
epoch: 18, loss: 0.59
epoch: 19, loss: 0.54
epoch: 20, loss: 0.50
epoch: 21, loss: 0.46
epoch: 22, loss: 0.42
epoch: 23, loss: 0.39
epoch: 24, loss: 0.36
epoch: 25, loss: 0.34
epoch: 26, loss: 0.31
epoch: 27, loss: 0.29
epoch: 28, loss: 0.27
epoch: 29, loss: 0.25
epoch: 30, loss: 0.23
epoch: 31, loss: 0.22
epoch: 32, loss: 0.20
epoch: 33, loss: 0.19
epoch: 34, loss: 0.18
epoch: 35, loss: 0.17
epoch: 36, loss: 0.16
epoch: 37, loss: 0.15
epoch: 38, loss: 0.14
epoch: 39, loss: 0.13
epoch: 40, loss: 0.12
epoch: 41, loss: 0.12
epoch: 42, loss: 0.11
epoch: 43, loss: 0.11
epoch: 44, loss: 0.10
epoch: 45, loss: 0.1

In [ ]:
# our loss has dropped to close to 0.
# but is it actually working? let's see

# let's first define a reverse vocabulary mapping (idx>word type)
rev_vocab = dict((idx, word) for (word, idx) in vocab.items())
boblikes = prefixes[0].unsqueeze(0)
logits = network(boblikes)
probs = nn.functional.softmax(logits, dim=1).squeeze()
argmax_idx = torch.argmax(probs).item()
print('given "bob likes", the model predicts "%s" with %0.4f probability' % (rev_vocab[argmax_idx], probs[argmax_idx]))



given "bob likes", the model predicts "sheep" with 0.9893 probability
